# defect-inspect — resolution retrain experiment

**Question.** `defect-inspect` trains at 320 and scores mAP@50 **0.7107** on its test split.
`defectvision` trains at 640, and when run at 320 scores **0.7440**. Does training at the
larger resolution and *deploying* at 320 actually help, or is that difference something else?

**Why both resolutions are trained here.** Comparing a new 640-trained model against the
existing `best.pt` would confound two variables: the input resolution *and* the ultralytics
version that produced each model. So this notebook trains **both** 320 and 640 in this same
environment, from the same seed, on the same split. The 320 run is the control, and it should
land near 0.711 — if it does not, the environment differs from the original and the comparison
is void.

**What makes this controlled.** The train/val/test split is pinned by an embedded manifest
taken from the actual on-disk split, verified to reproduce it exactly (1236 / 353 / 181
images, 438 test boxes). No re-randomisation, no seed drift across Python versions.

Runtime: **Session options → Accelerator → GPU T4 x2**. Expect roughly 15 min for 320 and
40 min for 640. Nothing here needs your Google Drive or any private repository.


In [ ]:
# This notebook was written for Colab, which works in /content. Kaggle does not
# have it, so create it once and leave every other path in the notebook alone.
import os
os.makedirs('/content', exist_ok=True)
os.chdir('/content')
print(os.getcwd())


## 1. Confirm a GPU is attached

If this prints `cpu`, stop and change the runtime — CPU training would take many hours.


In [ ]:
import torch
print('torch', torch.__version__)
print('device:', 'cuda -> ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
assert torch.cuda.is_available(), (
    'No GPU. In the Kaggle editor: Session options > Accelerator > GPU T4 x2, then rerun.')


## 2. Install ultralytics


In [ ]:
%pip -q install ultralytics
import ultralytics; print('ultralytics', ultralytics.__version__)


## 3. Fetch NEU-DET

Sparse partial clone of the public mirror: the upstream repository is 423 MB and only the
`IMAGES` and `ANNOTATIONS` trees are needed, so `--filter=blob:none --sparse` pulls no file
contents until they are checked out.

This is **NEU-DET**, the detection set with Pascal-VOC XML boxes — not NEU-CLS, which is
classification-only and cannot train a detector. The assertion below catches that mistake.


In [ ]:
import subprocess, pathlib
UP = 'https://github.com/siddhartamukherjee/NEU-DET-Steel-Surface-Defect-Detection.git'
raw = pathlib.Path('/content/NEU-DET-raw')
if not raw.exists():
    subprocess.run(['git','clone','--filter=blob:none','--sparse','--depth','1',UP,str(raw)], check=True)
    subprocess.run(['git','-C',str(raw),'sparse-checkout','set','IMAGES','ANNOTATIONS'], check=True)

imgs = sorted(raw.rglob('*.jpg'))
anns = sorted(raw.rglob('*.xml'))
print(f'{len(imgs)} images, {len(anns)} annotations')
assert anns, 'No XML annotations -> this is NEU-CLS, the wrong dataset.'
assert len(imgs) >= 1770, f'expected at least 1770 images, found {len(imgs)}'


## 4. Rebuild the exact split

The manifest below is the real split, gzipped and base64-encoded (5.9 KB). Using it rather
than re-running the stratified shuffle removes any dependence on how a given Python version
implements `random.shuffle`.


In [ ]:
import base64, gzip, json
MANIFEST_B64 = 'H4sIAOdDrWoC/22d3a7duA2F3yXXU8Be/u+rFIMgyKRtgCBTJJletOi798z2QY6+tb+7TdkWSZlckiWS+7/vfnz6/uPdX//27uO3D//5/PUf7+dpfvfLG0ViAhVQy0gt10it6GXdR2pDn/s2Ugf6PHDtGq9lmkDNoAJqBTXKmRnXco4U2G14bAfz6wA13jl2v4+jMHI6RzWvP7v+/PXjlz++f/796/s5c9Fb0QfppcjqbtlJbyvpI0VXf2exOy/Q94sY6CLD7rI0fZJeKW7OFF3sLz6/kv3GpzfqsnModl49SJYgJ99KiXX92fO/Pvz4+M9P319ewQbqGKkddx4TqPFapgPUNVLzBArPBcQOCl0+nOEnteC5FXeuuHMb1VvG37gwcl7GDkZp1/GubR6J8co+EsfI8xh7Psfbrkdvn3/8+PTb++9/fPv7h4+fXpxpscbVGk9pfDjbU6M9rm27NG4m0T3E1XhYmwl0mugP1KrGTNYmUsaYZ7YuV2uLNYriN+5242F8Tmu8RPZlkj4Xu1F0XOVZe2PS3WNu+/b7ly+ffvvL56/vv3/88OXlNTxmrqfWx5z81Kq3Pgb4qXXRexcV4YF7z62ntT587an1nLRVZThNhkzGLarxDW9PrXqrKnzPOU+tm/agwxB9l7ddPrVe1u9thN26mmar8Vrtne1mNYfd+UDJ7y8rkNeZ5jF6I72TXoqsxx9eMdJz0dX9Wv09tBnoB9yN9EWat6fphexS7PIY5oHeKG72en6v54+6ftT1k+Lnqv6vuv8q+S+qu/DxjYO981W9IN6vv7z78e3D569Y8o/raizHJ1JY8E9YnGNZPU9Yqk9Y8E8HqBMfEfzcwHMz7oQogZjhNYgZiBmIGbALxAyZ4+uGHz4QZYE+CwRb+BwEWyDYAlFw4wp2KxisvBNdrtBnhT64cSPFTzc8hks7v+owDDsU3yHlDlV3SHmgzwN9HtD8QJ8HJDvA4SAHKIRLJ5ifYH6C+QnmJ3q50MuFXi70cqGXi5++UAjfm/NFfvhOxYcvCH5Lb6DwOQt3DT6zgRyZeQ284TGBvQVmdIP3G0Xm0A0MqB3FrDvBHAgQIEDg2IFjB44dOHbgylnAYSEH7jdAPWxMwM2z8hrYrWAANw82Xu5p7o3ibgc4ABGygQO0g9cHXh94fXZ2gnHAJUBA4PTBZlEAAcHWUQAIOcgBzEmAOZw+J5gDAoINnZxgfoL5SX54XSS444SBBo6E22RAlQArxvsWzPuYmmDOsFgYLOYlbP/BeNeRKwwZlovZClY8SgODhj1vI0/YNkwblo2pDlaOaQ8THSx+H5nC+mH82PGEI8APxgvwAsyD8Ag4BPwB7gD7h/ljOoT1Y/6DJ8AR4AcwfNg9DB2/Rzawa0yImA8xHcLCL27D1U5q7YxOfbl2QqfaCZ1q53Sqjdmpdlqn2jmdzqJL2BKnxX26XuL2RvBc4szFfi721V16W7q6T41Gil2K3VLiL8Vv6X3s6n/pbe/uv9Spx9dSpzaK73X1SJc4a/dXL7/2re9P2pEu8erxrcTbSrzayL53CUe6d/mL/VajtZU4e/Hfi99e/GrrfN6Lf+2l38v88ZSh9Dv6VKLG56j+a3P+XuSPdOlXl89Stzb357MPQUq82v2/PwEGurj3Ec9V2l/F/ir2V7G/Svur2dchSZ3YFNl0ndAU7KVgLwV7KZhLwVwK11LAk8KxlCencO3+gBjp4lfdt/pP14td4VoK157Ot6r7grkUrN3fDiNd7Ar2UjDX0hbMpWAuBWtZ+/k+jit2JW3BVAqWUjCUgqEUDPVYFyrdnxsjXYNbqJS9+ytxCpWylzh1+1HiFIiljlJTR6kpkEuBXI7mV+IVqLUlFqjlLPEK1FKgdn+2jHSJUyCXPokt9gVqKVBLgVoK1FIgxqfLzZYCsVpslBPWUqNcsjywHLCWHeWO7KmcsZYY5Yq1oKj1Q/kh763FQzllHfhv9Sy1LYodl7vt7KmcrXyrXKuO2smn/KrcqryoVgblI+UDZdJl0XWGT6Fqzi7rLuO+ODQXZbzGs/vxjPvejH+jeC2gEDcw7aAQYTCdoMAbDBBEMNc1MJ8XUBBlRrDDDMFmCDZDsBmCgV0gWCBYeCcECwQLBEMgBPVZwG4BO0RF3N8sbxSDPsgA2uHGFexWsFvBbgW7lb1AO4RRzAjXuL9EflJ4bAO7Dew2sNv4HNhteMsbmG9gjsd2jMOOcdgh2A7mO3thdA2Y72B+oE8EkdyfH28UOBzggHgTWvQJhU4odILdibE9wRwxLHc01hsFftDuArsLul5gfoHdBXYX2AG34JQBVAVQdX9QvFEMatpAIToJXc6k0CWMODM7QeASXlAwYqFC5F5BVOAOXEnQJ24ErgS4EmBHgB1BwFgWRmmBHaAkgJKsjOeCZACPADwCuKCuG9ghWiuAkgA8ArjIxmAysMONAIgAEu6jhzcK7AAX2dknxhYAweC4A9oBOwJ8CPAhcPv7ROGNgtAAgQAEAhAI3P418Oln7AAIMAcIBCAQuH3g9vcZwr8QnPAzjg8LFpg0LBomDJuFkcJGMb/BQjGfwT7Hu2CBmMtgjZjJMHXBMDEfwfZgbLA1GBd+j6MGy4JhIf4TUxIs7hhHAOGPIxuYHmwNpoYJBoYGO7vGrmFYCH3F3AIbs/BDC9C0YL3XhXE3xhotmnKyYMzJQiwni7GcLJxyMn1MTG802TVmdDYxNRR0NjEtRFOZWyjo69K7G/VxC221d7FYl4t1uWiXprkFab4urLvRGFkg6utquxvtZawqkr2M1V6GdbmZQpsptGlIsUm0mUSbSWRdWlzufUjw1GgDt5uWuzE/7A0d1udhch4aJG2qW1Dx6xK+G23gT9P9NDlP0/004U8T3uKc1asvk12D4C8b48sUukyhyxSyyOv7nKJjtC3kXOPQZ2vUxy3m3PA9BuUxKI/BdgzRYrAdc+vMGllvIpmWGqxvCB1/3GSPiWlY/po60o02csbH8iZiU0FsKohlWNwnKU+NJrtNGq9fUp2TYHLaTJJV8xxMJMv6iM0P+jJsKohNBbG0hNcPsW5URjZIm4ppI2dK7prhYRLtxnw35jZpqG0emkliA2dTzn2W89Rosh82moeKZAppm8lu09DrB2Y3muw2N8VmnFj6kHq6zTixGSc248RmnNjk8vp92o0qpsi5WE6QtNl9MmiGO4vIYgAj3Rm62DLVVqmGN7YcNbSxxaiBjWHNJvIZ0himGKQYouzCdxe+BjK23DSIMYSR2wxKDDQMHmyVajhgy1G5zZDBMMAgwBDAFqe2NjWgsJWpNYl45vq21jQ0sJWmAYThw2UZW5p7pxluk9+qGXmTp//t2qoZeZojd288PCcQqryeQThrTp9m2d3bD8+tqsWsWsyqxaxaxNMgPQ9SJYtKFpUsKllUMh2yRcVd/F5P21RxFxV3UcFWffGrSrbqQK4qr2Yh3ueMz60q76rDu6oWymxT1TRt8z6tfG5VcTcVd1NxtYNdBdNszPsk87lVx3z3dF2VQXU4VLBDBTtUsEPH0VOZD5fB84j1teutp4p76iicKsLp/aoIlw7ZpTJoZu+9R/LcqkN2qRaXanGpFjqDaSbzpI3eOmurd7toq+ZoT577vWvroa2qroqrU2Bmv1eVUBiKTlXx4XXJPFc9KkM82V0l08kuOtlFJ7voZOeCLaqwToFZvAcVVye76GTnY6PzV1a/V0VYvT6ANuoobCrC5j2oCDonReek7F64QGXQKSU6pUTrQrg16UQTnWiiE020sMR9sv7cqsOgRSjuvY/nVrUGL02hM01Or/WgjSrYpYLphBCdEKITwqLFJqxRy1Lo4zZYWllDfVerbag/a/ULrZNhYqovqyvrmtP0Ue/WBad6vC5C1bPVsTcT05TU9aeigrq/+rnWPdlNIq0vYhIpHCga6KpTF50KELq4NH20Qo0ihgKG4oXCha5LdVmquKIIomtSXZIq1OgyVVepikkKSYpIVc2kSsNMVTpm6utVOmaq0jFTVaqZqpTMVKVqppKm2M0lztP1Eqcr48xVuWYu8apUzb0bMtIlXkqelDwp/lmLLnlS/NOFgIr/UvyrtM69uTHSxX/pQkLFfyl+9fjadKmzVvdrqdN1hEqbrbTZSput2G/Fbuu6RcW+bq+6QnPVFbpDJUa6xN+7v2K/N/sum1TGU3WM5qpjdH/8j3TJc5Q8R/E7a7jP4n8W/7P4Vx2l++t+pEvfs/g3WeJcxb7KMt0f8iNNdasI1FRk01UDaqoaUYVjKRxL4VgKxwoXUjiWua+X9IVjKUdMeVLKtFM4lhr7qi+WHq2n6yVOwVoK1lL1ypKuCFb8qqRWCsZSMHbHFox08VuaXw1HwVyqYFrWrlhW/KpA2h2UPVYwK7K6L5xL4Vy2LohW6hXOpXAuVZ/tjtQe6ZKvgCwFZCkgSwFZv93CqRROpXAqhVM1B6ZgKwVbKdjK2eXfSpyCrRRMpcnqvmDoPhEf6dLuojoUrhyh7L7MvKy8jLxsvEycQpe9V33Asv4y/rL9Mn3eW5X8yu7L7KvKYBl92XiZOO8te98pRU37NcuXb5Tpl+UfZFQTeM3f5QblBVUCsnyCF8sDygHK/sv8axIvZ6gpnELUhF1lFGu6LjcpL6kSiy/fJL/+8u7fH76wLDpqw7EKH0vmoWjZjDJlc1gQEIXcFtbkQ5U8lEOaUc9oRg2jeeOd6BMlie4V4FvtO1StA3MUsJpRx2c++Ry6RMmqibXVUJaK5c1Y6mpheTM8h0ryQZUoFnbDgAWlnYIhCoYoKM6UneXNICbKMAWVk3KyPhf6RO0i1JhDiTlUmEOBOdgQCsqhFByGCiqPF6ATyqShNBmKinX9kalLE9X1rm5TJTrmdK2hqpeydHGdrl5T17v4TBePKbKLv1Q1lqqDMF9d4KTrqXQBkq6IX2UeqhhNnkrm1/1dhqILLyxd46Pkq8oCWbsoRxVuqFpDqQIC2boISFfZ6LIWXVeirldmfa6+XgUZqlBC1VzgzWU5VZ+kypFUNZKqNlIlkKq4R9XyqNIdRXHAqy5HleUoa6wqGReT+0Egy3di9jvyepE0OC/MFmcyN3O0mTMNAunHyFCbkYk2n0xbBgP+jQLzEJE3zFxkJvkydZd5tszPZRYsCOav4rGNKaRIuUT+eZD2F2ScB+l9OfgckjqR1ZeTqZq4huELcvGQi4yiBhgTJBQjZxh2AbNABi/+tGG8CyODvFvkzyJ9FvmyyEbEhfH5S3P1rM3yVTRjTfOhLOR8tiDpedEsOEuhWfQfJzRnzBJ4Nk28svwfC6N9LavQjabmrulUxujQJCnNPrIIb0vmzKT/RGFtFp0+a2qMhcZrYP6iyS1256rpFMZ909QJ08jeWyzWOfYyY6HIsQDgWARwLD7XUngsg8eyZSwvRoSxZBUbMstQtGGwBBIzXrNdy+qwtAzLoLBkCQ8zmzz22KN2NYRVO4jHpXr8qIdTKjc9Cp83D+nUGLxdue0eMuh/SOIxeNrooV8acaEBiln8r0c84EhjQTQ8NxqeGw3EjYbcRgMdoq8tGqwQfUEeZ6ZRCNEo2Ogbjr7haDBANNjUg9o0gDR69p/Tu9V+1Vuj5+rRg/Vo8KaG0GmsnJqNWo2+cnVJDUfTWDJ9hxpJprFd+lbqxK5jA+pAburD9j78rrPmOn5c+jC5uNdm8ly7vPPex6l1/ajjyv7TnSZL+rNOV2v7c64tzbn2/eeKu8jUB5Z1bDH1GVsde/QxSZ9b9J/sFFm91cl91j5S6iOjIqu7Gvscfb2PjPrMp+jaeq6whzoerP8XqqCF0rQUraO3OmmrCIE6JyuKKtehWA1AnXHVkVQdQb1Y1q//+z+xZGtyR3QAAA=='
SPLIT = json.loads(gzip.decompress(base64.b64decode(MANIFEST_B64)))
for k, v in SPLIT.items():
    print(f'{k}: {len(v)}')
assert (len(SPLIT['train']), len(SPLIT['val']), len(SPLIT['test'])) == (1236, 353, 181)


### VOC → YOLO

Same conversion as `scripts/prepare_dataset.py`, including its self-check, so the label files
are byte-comparable with the originals. Class indices come from the alphabetically sorted
class names, which is what fixes 0=crazing … 5=scratches.


In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path
import shutil

CLASSES = ['crazing','inclusion','patches','pitted_surface','rolled-in_scale','scratches']
IDX = {c: i for i, c in enumerate(CLASSES)}

def to_yolo(box, w, h):
    xmin, ymin, xmax, ymax = box
    xmin, xmax = sorted((xmin, xmax))
    ymin, ymax = sorted((ymin, ymax))
    return ((xmin+xmax)/2/w, (ymin+ymax)/2/h, (xmax-xmin)/w, (ymax-ymin)/h)

# The same assertions prepare_dataset.py carries.
assert to_yolo((0,0,100,200), 200, 200) == (0.25, 0.5, 0.5, 1.0)
assert to_yolo((100,200,0,0), 200, 200) == (0.25, 0.5, 0.5, 1.0)

img_by_stem = {p.stem: p for p in imgs}
ann_by_stem = {p.stem: p for p in anns}
root = Path('/content/yolo')
if root.exists(): shutil.rmtree(root)

counts = {}
for split, stems in SPLIT.items():
    (root/split/'images').mkdir(parents=True, exist_ok=True)
    (root/split/'labels').mkdir(parents=True, exist_ok=True)
    boxes = 0
    for stem in stems:
        shutil.copy(img_by_stem[stem], root/split/'images'/f'{stem}.jpg')
        r = ET.parse(ann_by_stem[stem]).getroot()
        sz = r.find('size'); W, H = int(sz.find('width').text), int(sz.find('height').text)
        lines = []
        for o in r.findall('object'):
            name = o.find('name').text.strip()
            b = o.find('bndbox')
            xc, yc, bw, bh = to_yolo(tuple(float(b.find(k).text) for k in ('xmin','ymin','xmax','ymax')), W, H)
            lines.append(f'{IDX[name]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')
        (root/split/'labels'/f'{stem}.txt').write_text('\n'.join(lines) + '\n')
        boxes += len(lines)
    counts[split] = boxes
    print(f'{split}: {len(stems)} images, {boxes} boxes')

# Box counts from the original split. A mismatch means the dataset or the
# conversion differs, and any comparison downstream would be meaningless.
assert counts == {'train': 2887, 'val': 801, 'test': 438}, counts
print('\nsplit matches the original exactly')


In [ ]:
yaml_text = '''train: train/images
val: val/images
test: test/images
names:
  0: crazing
  1: inclusion
  2: patches
  3: pitted_surface
  4: rolled-in_scale
  5: scratches
'''
(root/'data.yaml').write_text(yaml_text)
DATA = str(root/'data.yaml')
print(DATA)


## 5. Train both resolutions

Identical settings apart from `imgsz`: YOLOv8n from COCO weights, 100 epochs, patience 25,
seed 42, `deterministic=True`. The 320 run is the control.


In [ ]:
from ultralytics import YOLO
import time

def train(imgsz):
    t0 = time.time()
    m = YOLO('yolov8n.pt')
    m.train(data=DATA, imgsz=imgsz, epochs=100, patience=25, seed=42,
            deterministic=True, device=0, project='/content/runs',
            name=f'train{imgsz}', exist_ok=True, verbose=False)
    print(f'imgsz {imgsz}: trained in {(time.time()-t0)/60:.1f} min')
    return f'/content/runs/train{imgsz}/weights/best.pt'

W320 = train(320)


In [ ]:
W640 = train(640)


## 6. Evaluate every combination on the test split

Four numbers: each model at each resolution. `conf=0.001` is a metric sweep, not the
operating threshold.


In [ ]:
import itertools, pandas as pd

rows = []
for (tag, w), ev in itertools.product([('trained320', W320), ('trained640', W640)], [320, 640]):
    r = YOLO(w).val(data=DATA, split='test', imgsz=ev, conf=0.001, device=0, verbose=False)
    b = r.box
    rows.append({'model': tag, 'eval_imgsz': ev, 'P': round(b.mp,4),
                 'R': round(b.mr,4), 'mAP50': round(b.map50,4), 'mAP50-95': round(b.map,4)})
    print(rows[-1])

df = pd.DataFrame(rows)
print()
print(df.to_string(index=False))
df.to_csv('/content/resolution_matrix.csv', index=False)


### Read the result

- **Control check.** `trained320 @ 320` should be close to **0.7107**. If it is far off, the
  environment differs from the one that made the original weights and nothing below is
  comparable — say so rather than reporting the numbers.
- **The hypothesis.** It holds only if `trained640 @ 320` exceeds `trained320 @ 320` on the
  *same* split. That is the comparison the earlier cross-project result could not make,
  because the two projects used different splits.
- **Expect** `trained320 @ 640` to be far worse (the earlier measurement: 0.7107 → 0.3091).
  Its presence here is a sanity check that the asymmetry reproduces.


In [ ]:
piv = df.pivot(index='model', columns='eval_imgsz', values='mAP50')
print(piv.to_string())
ctrl = piv.loc['trained320', 320]
print(f'\ncontrol trained320@320 = {ctrl:.4f}  (original: 0.7107)')
if abs(ctrl - 0.7107) > 0.03:
    print('CONTROL FAILED -- environment differs from the original; do not compare.')
else:
    d = piv.loc['trained640', 320] - ctrl
    print(f'control OK. train-at-640 then deploy-at-320 changes mAP50 by {d:+.4f}')
    print('hypothesis SUPPORTED' if d > 0 else 'hypothesis NOT supported')


## 7. Export for the Pi, and download

Exports the 640-trained model to ONNX at **320** input — the deployment resolution — plus
FP32 and INT8, matching what `scripts/export_onnx.py` produces. Static 320 input because the
Pi runtime fixes the shape.


In [ ]:
best = YOLO(W640)
best.export(format='onnx', imgsz=320, opset=12, simplify=True, dynamic=False)
print('exported')


In [ ]:
import shutil, os
# /kaggle/working is what `kaggle kernels output` retrieves, so everything
# worth keeping is written there rather than pushed at a browser.
OUT = '/kaggle/working'
for s_, d_ in [(W320, 'best_trained320.pt'), (W640, 'best_trained640.pt'),
               (W640.replace('.pt', '.onnx'), 'best_trained640_320input.onnx'),
               ('/content/resolution_matrix.csv', 'resolution_matrix.csv')]:
    if os.path.exists(s_): shutil.copy(s_, f'{OUT}/{d_}')
for d in ['/content/runs/train320', '/content/runs/train640']:
    if os.path.exists(d):
        shutil.copytree(d, f'{OUT}/{os.path.basename(d)}_run', dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns('*.pt'))
print(sorted(os.listdir(OUT)))


---

## What to do with the download

Unzip into the repository and tell Claude it is there. What matters:

- `resolution_matrix.csv` — the four-cell result, the thing the paper needs.
- `best_trained640.pt` — the candidate replacement weights, **if** the hypothesis held.
- `best_trained640_320input.onnx` — ready to quantise to INT8 and benchmark on the Pi.
- `train*_run/` — curves and confusion matrices for both runs, minus checkpoints.

Do not swap the deployed model on the strength of these numbers alone. The Pi is currently
under-volting, so re-benchmark latency only after the supply is fixed; accuracy here is
unaffected by that, since it was measured on this GPU.
